# Field Shift: Member 3 IoT Soil Monitoring & Ground Sensor Subsystem
## Comprehensive Scientific, Technical & Calibration Walkthrough

- **Project:** Field Shift — Adapting Farms with NASA Data (AgriSense Fusion)
- **Study Area:** Godagari Upazila, Rajshahi District, Bangladesh (AEZ-26 High Barind Tract)
- **Centroid:** 24.469°N, 88.330°E | **CRS:** EPSG:32645
- **Subsystem Role:** Ground-truth IoT sensor data modeling, ESP32 telemetry, soil calibration, and integration with NASA SMAP and ISRIC SoilGrids
- **Status:** Prototype Design & Simulated Calibration Benchmark (Hardware Deployment Pending)

### 1. Environment Setup and Library Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

### 2. Load Processed Datasets & Audit Data Schemas

In [ ]:
# Path definitions relative to Member 3 directory
BASE_DIR = os.path.dirname(os.path.abspath('')) if 'notebooks' in os.getcwd() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')
SIM_DIR = os.path.join(BASE_DIR, 'data', 'simulated')
CAL_DIR = os.path.join(BASE_DIR, 'calibration')

sensor_df = pd.read_csv(os.path.join(DATA_DIR, 'soil_sensor_dataset.csv'))
daily_df = pd.read_csv(os.path.join(BASE_DIR, 'analysis', 'daily_sensor_summary.csv'))
soilgrids_df = pd.read_csv(os.path.join(DATA_DIR, 'soilgrids_properties.csv'))
smap_df = pd.read_csv(os.path.join(DATA_DIR, 'smap_soil_moisture.csv'))
cal_df = pd.read_csv(os.path.join(CAL_DIR, 'calibration_template.csv'))

print(f"Sensor Readings Dataset Shape: {sensor_df.shape}")
print(f"Daily Summary Dataset Shape:  {daily_df.shape}")
print(f"SoilGrids Profiles Shape:     {soilgrids_df.shape}")
print(f"SMAP Daily Series Shape:      {smap_df.shape}")
print(f"Calibration Points:           {len(cal_df)}")

sensor_df.head(3)

### 3. Data Integrity and Missingness Audit

In [ ]:
print("--- Missing Value Audit ---")
print(sensor_df.isnull().sum())

print("\n--- Operational Metadata Separation ---")
print(sensor_df['source'].value_counts())
print(sensor_df['is_simulated'].value_counts())
print("\nHardware Fault Flags Injected:")
print(sensor_df['status_flags'].value_counts())

### 4. Soil Sensor Calibration Verification (Linear vs. Quadratic Models)

Evaluates the ADC-to-VWC conversion equations against laboratory gravimetric reference standards.

In [ ]:
adc = cal_df['raw_adc_mean'].values
vwc = cal_df['reference_vwc_percent'].values / 100.0

# Linear regression
p1 = np.polyfit(adc, vwc, 1)
pred_lin = np.polyval(p1, adc)
r2_lin = 1 - np.sum((vwc - pred_lin)**2) / np.sum((vwc - np.mean(vwc))**2)
rmse_lin = np.sqrt(np.mean((vwc - pred_lin)**2))

# Quadratic regression
p2 = np.polyfit(adc, vwc, 2)
pred_quad = np.polyval(p2, adc)
r2_quad = 1 - np.sum((vwc - pred_quad)**2) / np.sum((vwc - np.mean(vwc))**2)
rmse_quad = np.sqrt(np.mean((vwc - pred_quad)**2))

print(f"Linear Model:    VWC = {p1[0]:.6f} * ADC + {p1[1]:.5f}")
print(f"                 R² = {r2_lin:.4f}, RMSE = {rmse_lin*100:.2f} % VWC")
print(f"Quadratic Model: VWC = {p2[0]:.5e} * ADC² + {p2[1]:.6f} * ADC + {p2[2]:.5f}")
print(f"                 R² = {r2_quad:.4f}, RMSE = {rmse_quad*100:.2f} % VWC")

# Plot calibration curve
plt.figure(figsize=(8, 5))
x_dense = np.linspace(1500, 3600, 200)
plt.scatter(adc, vwc * 100, color='crimson', s=60, zorder=5, label='Laboratory Core Samples (ASTM D2216)')
plt.plot(x_dense, np.polyval(p1, x_dense) * 100, 'b--', label=f'Linear Fit (R² = {r2_lin:.4f})')
plt.plot(x_dense, np.polyval(p2, x_dense) * 100, 'darkgreen', label=f'Quadratic Fit (R² = {r2_quad:.4f})')
plt.axhspan(16.5, 24.4, color='orange', alpha=0.15, label='Readily Available Water (RAW: 16.5% - 24.4%)')
plt.axhline(16.5, color='red', linestyle=':', label='Permanent Wilting Point (PWP = 16.5%)')
plt.axhline(32.3, color='blue', linestyle=':', label='Field Capacity (FC = 32.3%)')
plt.xlabel('ESP32 Raw ADC Value (12-bit, 0-4095)')
plt.ylabel('Volumetric Water Content (VWC %)')
plt.title('Capacitive Soil Moisture Calibration Curve — Godagari Silty Clay Loam')
plt.legend(frameon=True, fontsize=8)
plt.tight_layout()
plt.show()

### 5. Time Series Analysis: Soil Moisture Depletion & Diurnal Thermal Dynamics

In [ ]:
sensor_df['timestamp'] = pd.to_datetime(sensor_df['timestamp'])
valid_df = sensor_df[sensor_df['status_flags'] == 'NORMAL_OPERATION']

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Panel 1: Soil Moisture VWC%
ax[0].plot(valid_df['timestamp'], valid_df['soil_moisture_vwc'], color='teal', lw=1.2, label='In-Situ VWC (10 cm depth)')
ax[0].axhline(32.3, color='blue', linestyle='--', label='Field Capacity (32.3%)')
ax[0].axhline(24.4, color='darkorange', linestyle='--', label='Irrigation Trigger (24.4%)')
ax[0].axhline(16.5, color='crimson', linestyle='--', label='Permanent Wilting Point (16.5%)')
ax[0].set_ylabel('VWC (%)')
ax[0].set_title('Godagari Field Node: 60-Day Soil Moisture Dynamics (Drying Cycles & Irrigation Spikes)')
ax[0].legend(loc='upper right', frameon=True)

# Panel 2: Soil vs Air Temperature
ax[1].plot(valid_df['timestamp'], valid_df['air_temperature'], color='coral', lw=0.8, alpha=0.7, label='Air Temp (°C) [High Amplitude]')
ax[1].plot(valid_df['timestamp'], valid_df['soil_temperature'], color='saddlebrown', lw=1.2, label='Soil Temp 10 cm (°C) [Thermal Damping]')
ax[1].set_ylabel('Temperature (°C)')
ax[1].set_xlabel('Timestamp (UTC)')
ax[1].set_title('Thermal Buffering in Barind Clay (Thermal Lag ~2.5 hrs)')
ax[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

### 6. SoilGrids Depth Profiles (Barind Heavy Clay & Plow Pan Hardening)

In [ ]:
print("ISRIC SoilGrids v2.0 Profile for Godagari Centroid (24.469°N, 88.330°E):")
soilgrids_df[['depth_interval', 'depth_label', 'clay_percent', 'bulk_density_g_cm3', 'ph_water', 'soc_g_kg']]

### 7. Run Unit Tests Inside Notebook

In [ ]:
import unittest
test_loader = unittest.TestLoader()
test_suite = test_loader.discover(os.path.join(BASE_DIR, 'tests'), pattern='test_*.py')
test_runner = unittest.TextTestRunner(verbosity=2)
test_result = test_runner.run(test_suite)
print(f"\nTotal Tests Run: {test_result.testsRun}")
print(f"Failures: {len(test_result.failures)}")
print(f"Errors:   {len(test_result.errors)}")
if test_result.wasSuccessful():
    print("\nSUCCESS: All unit tests passed cleanly!")